In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import shap
import matplotlib.pyplot as plt
import seaborn as sns

# ---- Load Data ----
file_path = "shap/rmsf_input_shap5.csv"
df = pd.read_csv(file_path)

catalytic_cols = [col for col in df.columns if col.startswith("Cat_")]
substrate_cols = [col for col in df.columns if col.startswith("Sub_")]

X = df[catalytic_cols].values  # shape: (6, N)
Y = df[substrate_cols].values  # shape: (6, M)

shap_matrix = []

for i in range(Y.shape[1]):
    y = Y[:, i]
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X, y)

    explainer = shap.Explainer(model, X)
    shap_values = explainer(X)

    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)  # shape: (num_cat_res,)
    shap_matrix.append(mean_abs_shap)

shap_df = pd.DataFrame(shap_matrix, columns=catalytic_cols, index=substrate_cols)

mean_shap_overall = shap_df.mean(axis=0)
top10_residues = mean_shap_overall.sort_values(ascending=False).head(10)

print("Top 10 catalytic residues by mean SHAP across all substrate residues:")
print(top10_residues)

plt.figure(figsize=(10, 6))
ax = sns.heatmap(shap_df, annot=True, cmap="YlGnBu", cbar_kws={'label': 'Mean SHAP Value'})

# Rotate annotation texts inside heatmap cells
for text in ax.texts:
    text.set_rotation(90)

plt.title("SHAP Feature Importance: Catalytic → Substrate RMSF")
plt.xlabel("Catalytic Residues")
plt.ylabel("Substrate Residues")
plt.tight_layout()
plt.show()

# ---- Plot bar chart of top 10 residues ----
plt.figure(figsize=(8, 5))
top10_residues.plot(kind='bar')
plt.ylabel("Mean SHAP Value")
plt.title("Top 10 Catalytic Residues by SHAP Importance (Overall)")
plt.tight_layout()
plt.show()


In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
import shap
import seaborn as sns
import matplotlib.pyplot as plt
import os

# === Create output folder ===
output_folder = "shap_results"
os.makedirs(output_folder, exist_ok=True)

# === Extract substrate features and catalytic targets ===
substrate_cols = [col for col in full_df.columns if col.startswith("Sub_")]
catalytic_cols = [col for col in full_df.columns if col.startswith("Cat_")]

X = full_df[substrate_cols].values  # Substrate features
Y = full_df[catalytic_cols].values  # Catalytic targets

# === Train models and compute SHAP values ===
shap_matrix = []

for i in range(Y.shape[1]):  # For each catalytic residue
    y = Y[:, i]
    model = RandomForestRegressor(n_estimators=100, random_state=0)
    model.fit(X, y)

    explainer = shap.Explainer(model, X)
    shap_values = explainer(X)

    mean_abs_shap = np.abs(shap_values.values).mean(axis=0)
    shap_matrix.append(mean_abs_shap)

# === Convert to DataFrame ===
shap_df = pd.DataFrame(shap_matrix, columns=substrate_cols, index=catalytic_cols)

# === Save SHAP matrix as CSV ===
shap_csv_path = os.path.join(output_folder, "shap_matrix_substrate_to_catalytic.csv")
shap_df.to_csv(shap_csv_path)
print(f"SHAP matrix saved to: {shap_csv_path}")

# === Plot heatmap and save ===
plt.figure(figsize=(10, 6))
sns.heatmap(shap_df, annot=True, cmap="YlGnBu", cbar_kws={'label': 'Mean SHAP Value'})
plt.title("SHAP Feature Importance: Substrate → Catalytic RMSF (All Structures)")
plt.xlabel("Substrate Residues")
plt.ylabel("Catalytic Residues")
plt.tight_layout()
heatmap_path = os.path.join(output_folder, "shap_heatmap_substrate_to_catalytic.png")
plt.savefig(heatmap_path, dpi=300)
plt.close()
print(f"Heatmap saved to: {heatmap_path}")

# === Optional: Top 10 substrate residues overall ===
top10_substrates = shap_df.mean(axis=0).sort_values(ascending=False).head(10)
top10_path = os.path.join(output_folder, "top10_substrate_residues.csv")
top10_substrates.to_csv(top10_path)
print(f"Top 10 substrate residues saved to: {top10_path}")

# === Optional: Bar plot of top 10 substrates ===
plt.figure(figsize=(8, 5))
top10_substrates.plot(kind='bar')
plt.ylabel("Mean SHAP Value")
plt.title("Top 10 Substrate Residues by SHAP Importance")
plt.tight_layout()
barplot_path = os.path.join(output_folder, "top10_substrate_barplot.png")
plt.savefig(barplot_path, dpi=300)
plt.close()
print(f"Top 10 bar plot saved to: {barplot_path}")
